# OIBSIP — Data Analytics Level 1
## EDA on Retail Sales Data

**Goal:** Explore retail transactions, identify sales trends, compare product categories, examine customer demographics, and write evidence-based recommendations.

**Dataset:** `data/retail_sales_dataset.csv`

> Run the notebook from top to bottom. Review every result and chart before writing your final conclusions. This notebook is tailored to the columns in the supplied dataset.

## 1. Import libraries

If a library is missing, install it in a notebook cell with `%pip install pandas numpy matplotlib seaborn`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("data/retail_sales_dataset.csv")
if not DATA_PATH.exists():
    # fallback when the notebook is run from a different working directory
    DATA_PATH = Path("../data/retail_sales_dataset.csv")

df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

## 2. Understand the dataset
Check column names, data types, date range, unique values, and summary statistics.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nDataset info:")
df.info()

print("\nNumeric summary:")
display(df.describe(include="all").T)

## 3. Data quality checks
Check missing values, exact duplicates, duplicated transaction IDs, and whether total amount matches quantity × price per unit.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_report)

print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Transaction IDs:", df["Transaction ID"].duplicated().sum())

# Convert data types
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
numeric_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

formula_check = df["Quantity"] * df["Price per Unit"]
mismatch = df[df["Total Amount"].ne(formula_check)]
print("Rows where Total Amount != Quantity × Price per Unit:", len(mismatch))
display(mismatch.head())

# This dataset has no missing values according to the initial check. If that changes,
# choose a treatment based on context rather than filling or dropping values blindly.
df_clean = df.drop_duplicates().copy()
print("Rows after removing exact duplicates:", len(df_clean))

## 4. Create useful time features
Create year, month, month name, and quarter. This makes monthly and quarterly comparisons easier.

In [ ]:
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.to_period("M").astype(str)
df_clean["Month Name"] = df_clean["Date"].dt.month_name()
df_clean["Quarter"] = df_clean["Date"].dt.to_period("Q").astype(str)

display(df_clean[["Date", "Year", "Month", "Quarter"]].head())
print("Date range:", df_clean["Date"].min(), "to", df_clean["Date"].max())

## 5. Overall KPIs
Calculate total sales, transaction count, units sold, average transaction amount, unique customers, and average customer age.

In [ ]:
kpis = pd.Series({
    "Total Sales": df_clean["Total Amount"].sum(),
    "Number of Transactions": df_clean["Transaction ID"].nunique(),
    "Units Sold": df_clean["Quantity"].sum(),
    "Average Transaction Amount": df_clean["Total Amount"].mean(),
    "Median Transaction Amount": df_clean["Total Amount"].median(),
    "Unique Customers": df_clean["Customer ID"].nunique(),
    "Average Customer Age": df_clean["Age"].mean(),
})
display(kpis.to_frame("Value").round(2))

## 6. Monthly sales trend
Group total sales by month and visualize the trend. Look for high/low months, but be careful when the date range covers only part of a year.

In [ ]:
monthly_sales = (
    df_clean.dropna(subset=["Date"])
    .groupby("Month", as_index=False)["Total Amount"].sum()
    .sort_values("Month")
)
display(monthly_sales)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="Month", y="Total Amount", marker="o")
plt.title("Monthly Retail Sales")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

if not monthly_sales.empty:
    print("Highest sales month:", monthly_sales.loc[monthly_sales["Total Amount"].idxmax()].to_dict())
    print("Lowest sales month:", monthly_sales.loc[monthly_sales["Total Amount"].idxmin()].to_dict())

## 7. Quarterly sales comparison

In [ ]:
quarterly_sales = (
    df_clean.dropna(subset=["Date"])
    .groupby("Quarter", as_index=False)["Total Amount"].sum()
    .sort_values("Quarter")
)
display(quarterly_sales)

plt.figure(figsize=(9, 5))
sns.barplot(data=quarterly_sales, x="Quarter", y="Total Amount")
plt.title("Quarterly Retail Sales")
plt.xlabel("Quarter")
plt.ylabel("Total Sales")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Product category analysis
Compare total revenue, transaction count, and units sold by product category.

In [ ]:
category_summary = (
    df_clean.groupby("Product Category")
    .agg(
        Transactions=("Transaction ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Total_Sales=("Total Amount", "sum"),
        Average_Transaction=("Total Amount", "mean")
    )
    .sort_values("Total_Sales", ascending=False)
)
display(category_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary.reset_index(), x="Product Category", y="Total_Sales")
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary.reset_index(), x="Product Category", y="Units_Sold")
plt.title("Units Sold by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

## 9. Customer demographics
Explore gender distribution, sales by gender, and age distribution. These are descriptive comparisons, not proof that gender or age causes spending differences.

In [ ]:
gender_summary = (
    df_clean.groupby("Gender")
    .agg(
        Transactions=("Transaction ID", "nunique"),
        Units_Sold=("Quantity", "sum"),
        Total_Sales=("Total Amount", "sum"),
        Average_Transaction=("Total Amount", "mean")
    )
    .sort_values("Total_Sales", ascending=False)
)
display(gender_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=df_clean, x="Gender", ax=axes[0])
axes[0].set_title("Transactions by Gender")
sns.barplot(data=gender_summary.reset_index(), x="Gender", y="Total_Sales", ax=axes[1])
axes[1].set_title("Total Sales by Gender")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4.5))
sns.histplot(df_clean["Age"].dropna(), bins=15, kde=True)
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

age_bins = [17, 24, 34, 44, 54, 64, 100]
age_labels = ["18–24", "25–34", "35–44", "45–54", "55–64", "65+"]
df_clean["Age Group"] = pd.cut(df_clean["Age"], bins=age_bins, labels=age_labels)
age_summary = df_clean.groupby("Age Group", observed=False).agg(
    Transactions=("Transaction ID", "nunique"),
    Total_Sales=("Total Amount", "sum"),
    Average_Transaction=("Total Amount", "mean")
)
display(age_summary)

## 10. Correlation analysis
The correlation matrix shows linear relationships among numeric columns. Correlation does not establish causation; Total Amount is calculated from quantity and unit price in this dataset.

In [ ]:
numeric_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
corr = df_clean[numeric_cols].corr(numeric_only=True)
display(corr)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 11. Additional visualization: sales amount distribution

In [ ]:
plt.figure(figsize=(9, 4.5))
sns.histplot(df_clean["Total Amount"], bins=30, kde=True)
plt.title("Distribution of Transaction Amounts")
plt.xlabel("Transaction Amount")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

## 12. Write evidence-based insights
Use the outputs above to complete these observations. The cells below generate initial values, but you should interpret them and write recommendations in your own words.

In [ ]:
top_category = category_summary["Total_Sales"].idxmax()
top_category_sales = category_summary.loc[top_category, "Total_Sales"]
top_month = monthly_sales.loc[monthly_sales["Total Amount"].idxmax()]
top_gender = gender_summary["Total_Sales"].idxmax()

print(f"1. Highest-sales category: {top_category} ({top_category_sales:,.2f}).")
print(f"2. Highest-sales month in the available data: {top_month['Month']} ({top_month['Total Amount']:,.2f}).")
print(f"3. Gender group with the highest total sales: {top_gender} ({gender_summary.loc[top_gender, 'Total_Sales']:,.2f}).")
print("\nNow add at least three thoughtful insights and three recommendations to your project README.")

## 13. Export results for submission

This saves the main summary tables as CSV files. Save selected charts from the notebook as screenshots or export them separately for your README/demo.

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

missing_report.to_csv(OUTPUT_DIR / "missing_values_report.csv")
category_summary.to_csv(OUTPUT_DIR / "sales_by_category.csv")
gender_summary.to_csv(OUTPUT_DIR / "sales_by_gender.csv")
monthly_sales.to_csv(OUTPUT_DIR / "monthly_sales.csv", index=False)
quarterly_sales.to_csv(OUTPUT_DIR / "quarterly_sales.csv", index=False)
age_summary.to_csv(OUTPUT_DIR / "sales_by_age_group.csv")
corr.to_csv(OUTPUT_DIR / "correlation_matrix.csv")
df_clean.to_csv(OUTPUT_DIR / "retail_sales_cleaned.csv", index=False)

print("Saved summary tables and cleaned data to:", OUTPUT_DIR.resolve())

## 14. Final checklist

- [ ] All notebook cells run without errors.
- [ ] Data quality checks are documented.
- [ ] Monthly and quarterly sales trends are interpreted.
- [ ] Product category performance is compared.
- [ ] Customer demographics are explored where appropriate.
- [ ] Correlation heatmap and at least one additional visualization are included.
- [ ] At least three evidence-based insights and three practical recommendations are written.
- [ ] README explains the project, dataset source, tools, process, key findings, and limitations.
- [ ] GitHub repository contains the notebook and selected outputs, respecting dataset license/terms.
- [ ] Demo video shows the project running and the key findings.